# Datakwaliteit Archeologische Complexenbestand
auteur: Thya van den Berg

Organisatie: Rijksdienst voor het cultureel erfgoed

Programma: DNA-NL

Project: Bronnen- en datakwaliteit

Projectleider: Maurice de Kleijn

In dit notebook wordt het bestand DEF_complexen_v8 beoordeeld op datakwaliteit, volgens de datakwaliteitsrichtlijnen van Bronnen- en Datakwaliteit. Dit staat naast de beoordeling en suggesties die gedaan zijn vanuit de afdeling Archeologie in het bestand "reparatie en homogenisering complexenbestand". DEF_complexen_v8 bevat archaeologische complexen welke geassocieerd zijn met archeologische rijksmonumenten. Het is een kennisbestand dat geen verdere wettelijke waarde heeft, maar het heeft wel identificerende waarde, omdat hierin de werkelijk aanwezige archeologie binnen- of direct geassocieerd aan een rijksmonument wordt omschreven.

## Installeer packages (niet relevant voor non-programmeerder)

In [64]:
# %pip install duckdb
# %pip install matplotlib
# %pip install mpl_toolkits
# %pip install shapely

## Laad packages (niet relevant voor non-programmeerder)

In [65]:
import duckdb
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
from shapely import wkt


## Benodigde variabelen (niet relevant voor non-programmeerder)

### Voor medallion architecture

In [66]:
### Variabelen definiëren voor Medallion Architecture
datalake_path = os.path.normpath("./data/")
zilveren_laag_path = "zilver"

### Voor opslaan

In [67]:
# voor tussendoor opslaan
save_directory = os.path.normpath(r"./data/saved_parquet_files//")
if not os.path.exists(save_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(save_directory)

# voor opslaan "quick wins"
zilver_directory = os.path.normpath(f"./data/{zilveren_laag_path}//")
if not os.path.exists(zilver_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(zilver_directory)

## Benodigde functies (niet relevant voor non-programmeerder)

In [68]:
# sla output van een query op (template)
# file_name = 'query_output.parquet'
# duckdb.sql(f"""
#             COPY
#                 (SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER)
#                 TO '{os.path.join(save_directory, file_name)}'
#                 (FORMAT parquet);""")
#
# # alternatief
# duckdb.sql(f"""SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER""").write_parquet(os.path.join(save_directory, file_name))

def empty_duckdb_memory():
    """Empty the in-memory duckdb database"""
    schemas = duckdb.sql("""
                SELECT schema_name
                FROM information_schema.schemata
                WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'temp', 'main')""").fetchdf()
    for schema in schemas['schema_name']:
        tables = duckdb.sql(f"""
                        SELECT table_name
                        FROM information_schema.tables
                        WHERE table_schema = '{schema}'
        """).fetchdf()
        for table in tables['table_name']:
            duckdb.sql(f"DROP TABLE IF EXISTS {schema}.{table}")
        duckdb.sql(f"DROP SCHEMA IF EXISTS {schema}")


def save_complete_duckdb_memory(save_directory):
    """saves entire in-memory duckdb database to the given save_directory"""
    duckdb.sql(f"""EXPORT DATABASE '{save_directory}' (FORMAT parquet)""")

def load_duckdbset_to_memory(load_directory):
    """Loads an entire saved set of parquet files from the included "schema.sql" and "load.sql" files."""
    file_name = 'schema.sql'
    with open(os.path.join(load_directory, file_name), 'r') as schema:
        duckdb.sql(schema.read())

    file_name = 'load.sql'
    with open(os.path.join(load_directory, file_name), 'r') as load:
        duckdb.sql(load.read())
    duckdb.sql("SHOW SCHEMAS;")

## Duckdb setup (niet relevant voor non-programmeerder)
duckdb is een databasemanagementsoftware vergelijkbaar met PostGRES en MySQL. Het gebruikt SQL queries om met de data te interacteren. In dit notebook gaat dit volledig lokaal en in het werkgeheugen van de computer, anders dan in het DAP, echter, de queries die moeten worden uitgevoerd voor het verbeteren van de data zijn hetzelfde.

### Laad de Spatial Extension
https://duckdb.org/docs/current/core_extensions/spatial/overview

In [69]:
duckdb.sql("""
           INSTALL spatial;
           LOAD spatial;
           """)

### laad database (schema's) in de bronzen laag


In [70]:
duckdb.sql("""
           CREATE SCHEMA IF NOT EXISTS brons_archeologische_complexen;
           """)

### laad parquet files als een tabellen in de bronzen laag


In [71]:
duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_archeologische_complexen.def_complexen_v8 AS
           FROM read_parquet('./data/brons/def_complexen_v8/def_complexen_v8.parquet');
           """)


## Algemene omschrijvingen

Hier wordt de data in het algemeen uiteengezet. Dit zijn geen testen, maar er valt veel informatie terug te lezen over de opzet van de dataset.

### Omschrijf de eigenschappen van de kolommen.

In [72]:
df = duckdb.sql("""
        DESCRIBE brons_archeologische_complexen.def_complexen_v8;
        """).fetchdf()


### Omschrijf de eigenschappen van de waarden in de kolommen.
Let op: veel van deze omschrijvingen zijn niet erg behulpzaam voor tekstkolommen. Dit zijn: min, max, avg, std, q25, q50 en q75.

In [73]:
duckdb.sql("""
        SUMMARIZE brons_archeologische_complexen.def_complexen_v8;""").fetchdf()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,complex_id,DOUBLE,20.0,50999.0,4482,14936.711100339311,20197.506587843258,1173.1558922558922,2332.3364877133054,16567.20975938507,4126,0.00
1,terreinnum,DOUBLE,42.0,20016.0,2300,2786.5399903053803,3964.6532419246237,671.9497071688043,1387.452600800246,1610.1471065440778,4126,0.00
2,rijksmonum,DOUBLE,31550.0,532469.0,1877,119046.30392632089,167622.54516092004,45475.557098765436,45788.64903107646,46174.0,4126,0.00
3,cma,VARCHAR,02G-026,geen CMA-nummer,1356,NaN,NaN,NaN,NaN,NaN,4126,6.62
4,cma_volgnr,VARCHAR,12A-001-01,?,1645,NaN,NaN,NaN,NaN,NaN,4126,55.84
5,cpx_monito,BIGINT,0,5220,3415,1746.8800290838585,1241.7678228908678,718,1538,2801,4126,0.00
6,cpx_jaar,VARCHAR,20150101000000.000,20170912000000.000,4,NaN,NaN,NaN,NaN,NaN,4126,92.46
7,cpx_code_b,VARCHAR,(Ring)walburg,weg,62,NaN,NaN,NaN,NaN,NaN,4126,0.00
8,cpx_code_o,VARCHAR,EGMW,VX,64,NaN,NaN,NaN,NaN,NaN,4126,0.51
9,cpx_code_n,VARCHAR,APVV,nederzettingsresten Swifterbantcultuur,76,NaN,NaN,NaN,NaN,NaN,4126,0.48


# Pas de datakwaliteits richtlijnen toe



In [74]:
# algemene variabelen
table =  "brons_archeologische_complexen.def_complexen_v8"
schema = "brons_archeologische_complexen"
table_name = 'def_complexen_v8'
dk_dimensies_voldaan = 0

## Dimensie 1: referentiële integriteit en unieke identificatie


- Vraag: Welke kolom bevat de primary key/ID?
    - Test A: Alle IDs in de primary key kolom zijn uniek.
    - Test B: Er zijn geen lege waarden in de primary key kolom.
    - Test C: In de databasestructuur is aangegeven dat de primary key kolom uniek moet zijn.
    - Test D: In de databasestructuur is aangegeven dat de primary key geen lege waarden mag bevatten.
    - Test E: In de databasestructuur is aangegeven dat de kolom een primary key is.
- Vraag: Welke kolom(men) bevat(ten) foreign key(s)?
    - Vraag: Naar welke dataset verwijst/verwijzen deze foreign key(s)?
        - Test F: Het is bekend van elke foreign key waar deze naar verwijst
    - Vraag: Is er een losse kolom voor elke foreign key? (note Thya: dit zou ik graag automatiseren en in een test veranderen, maar ik zie nog niet goed hoe.)
        - Test G: De foreign key(s) kom(t/en) overeen met een primary key uit de dataset waar deze naar verwijst.
    - Hebben alle foreign keys een eigen kolom?

#### Vraag: Welke kolom bevat de primary key?

In [75]:
primary_id_kolom = "complex_id"
test_resultaten_dim1 = []
print(f"de volgende kolom bevat de primary key: {primary_id_kolom}")

de volgende kolom bevat de primary key: complex_id


#### Test A

In [76]:
pk_is_unique = duckdb.sql(f"""
SELECT(
    SELECT
        COUNT()
    FROM (
        SELECT
            DISTINCT {primary_id_kolom}
        FROM {table}
    )
) == (
    SELECT
        COUNT()
    FROM {table}
)
AS unique_identifier;
""").fetchone()[0]
print(f"Test A :Alle IDs in de primary ID kolom zijn uniek: {pk_is_unique}")
test_resultaten_dim1.append(pk_is_unique)


Test A :Alle IDs in de primary ID kolom zijn uniek: True


#### Test B

In [77]:
pk_is_not_null = duckdb.sql(f"""
SELECT (
    SELECT
        COUNT()
    FROM {table}
    WHERE {primary_id_kolom} IS NULL
    ) = 0
AS null_identifier;
""").fetchone()[0]
print(f"Test B: Er zijn geen lege waarden in de primary id kolom: {pk_is_not_null}")
test_resultaten_dim1.append(pk_is_not_null)

Test B: Er zijn geen lege waarden in de primary id kolom: True


#### Test C

In [78]:
pk_schema_unique = duckdb.sql(f"""
SELECT
    CASE WHEN EXISTS(
        SELECT
            tc.constraint_type
        FROM
            information_schema.table_constraints tc
        JOIN
            information_schema.key_column_usage kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.table_schema = kcu.table_schema
            AND tc.table_name = kcu.table_name
        WHERE
            tc.table_schema = '{schema}'
            AND tc.table_name = '{table_name}'
            AND kcu.column_name = '{primary_id_kolom}'
            AND tc.constraint_type = 'UNIQUE')
    THEN True
    ELSE False
    END
AS has_unique_constraint;
""").fetchone()[0]
print(f"Test C: In de databasestructuur is aangegeven dat de primary key kolom unique moet zijn: {pk_schema_unique}")
test_resultaten_dim1.append(pk_schema_unique)

Test C: In de databasestructuur is aangegeven dat de primary key kolom unique moet zijn: False


#### Test D

In [79]:
pk_schema_not_null = duckdb.sql(f"""
SELECT(
    SELECT is_nullable FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table_name}' AND column_name = '{primary_id_kolom}'
    ) = 'NO' as not_nullable;""").fetchone()[0]
print(f"Test D: In de databasestructuur is aangegeven dat de primary key kolom niet null mag zijn: {pk_schema_not_null}")
test_resultaten_dim1.append(pk_schema_not_null)



Test D: In de databasestructuur is aangegeven dat de primary key kolom niet null mag zijn: False


#### Test E

In [80]:
pk_schema_exists = duckdb.sql(f"""
SELECT
    CASE WHEN EXISTS (
        SELECT
            1
        FROM
            information_schema.table_constraints tc
        JOIN
            information_schema.key_column_usage kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.table_schema = kcu.table_schema
            AND tc.table_name = kcu.table_name
        WHERE
            tc.table_schema = 'schema1'
            AND tc.table_name = 'table1'
            AND kcu.column_name = 'column1'
            AND tc.constraint_type = 'PRIMARY KEY'
    ) THEN TRUE
    ELSE FALSE
    END
AS is_primary_key;
""").fetchone()[0]
print(f"Test E: In de databasestructuur is aangegeven dat de primary key een primary key is: {pk_schema_exists}")
test_resultaten_dim1.append(pk_schema_exists)

Test E: In de databasestructuur is aangegeven dat de primary key een primary key is: False


#### Vragen: Welke kolom(men) bevat(ten) foreign key(s) ; Naar welke dataset verwijst/verwijzen deze foreign key(s)?
handmatig ingevoerd op basis van domeinkennis, dit is niet met code te achterhalen.
Let op: nog geen code geschreven om ook een kolom waar meerdere gemixte foreign keys in staan te checken, omdat dit voor deze dataset niet nodig is. Komt later. #TODO

In [81]:
# in de eerste waarde ([0]) van de lijst staat naar welke dataset/database verwezen wordt, in de tweede [1] staat het schema van hoe deze op dit moment is ingeladen in brons, in de derde [2] de tabelnaam in brons, en de vierde [3] de kolomnaam van de primary key van die dataset en de vijfde [4] de opgeslagen parquet file óf de waarde None als het niet om een vindbare dataset gaat.
dict_foreign_keys = {
    "terreinnum": ["DEF_terreinen_v5", "brons_archeologische_terreinen", "def_terreinen_v5",  "terreinnum", r"./data/brons/def_terreinen_v5/def_terreinen_v5.parquet"],
    "rijksmonum": ["rijksmonumentenregister", "brons_rijksmonumentenregister", "tblTEXT_OBJECT", "TXO_TEXT_KEY", r"./data/brons/rijksmonumentenregister/tblTEXT_OBJECT.parquet"],
    "cma":["Centraal_monumenten_archief", None, None, None, None], #zit in Proza, niet te testen. Dit is een gedigitaliseerd/gscand papieren archief. Geen verantwoordelijke, maar vraag Guide Mauro
    "cma_volgnr": ["Centraal_monumenten_archief", None, None, None, None], # zit in Proza, niet te testen
    "cpx_monito": [None,None, None, None, None] # Herkomst onbekend. In (verhouderde) data dictionary staat Unieke identifier RCE monitor, maar welke? Niet de BAAC nulmonitor en niet de huidige monitor.
}
for item in dict_foreign_keys:
    if dict_foreign_keys[item][0]:
        if dict_foreign_keys[item][4]:
            print(f"kolom {item} verwijst naar de primary key van {dict_foreign_keys[item][0]}, deze staat in kolom {dict_foreign_keys[item][3]} van {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}")
        else:
            print(f"kolom {item} verwijst naar een key van {dict_foreign_keys[item][0]}, deze dataset is niet ingeladen in de omgeving.")
    else:
        print(f"kolom {item} verwijst naar een onbekende dataset.")


kolom terreinnum verwijst naar de primary key van DEF_terreinen_v5, deze staat in kolom terreinnum van brons_archeologische_terreinen.def_terreinen_v5
kolom rijksmonum verwijst naar de primary key van rijksmonumentenregister, deze staat in kolom TXO_TEXT_KEY van brons_rijksmonumentenregister.tblTEXT_OBJECT
kolom cma verwijst naar een key van Centraal_monumenten_archief, deze dataset is niet ingeladen in de omgeving.
kolom cma_volgnr verwijst naar een key van Centraal_monumenten_archief, deze dataset is niet ingeladen in de omgeving.
kolom cpx_monito verwijst naar een onbekende dataset.


#### Test F

In [82]:
fk_ref_known = True

for item in dict_foreign_keys:
    if not dict_foreign_keys[item][0]:
        fk_ref_known = False
print(f"Test F: Het is voor elke foreign key bekend welke dataset deze naar refereert: {fk_ref_known}")
test_resultaten_dim1.append(fk_ref_known)

Test F: Het is voor elke foreign key bekend welke dataset deze naar refereert: False


#### Test G

laad de benodigde datasets in zodat er een vergelijking gemaakt kan worden

In [83]:
for item in dict_foreign_keys:
    if dict_foreign_keys[item][4]:
        duckdb.sql(f"""
                   CREATE SCHEMA IF NOT EXISTS {dict_foreign_keys[item][1]};
                   """)
        duckdb.sql(f"""
                   CREATE TABLE IF NOT EXISTS {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]} AS
                   FROM read_parquet('{dict_foreign_keys[item][4]}');
                   """)
    else:
        print(f"LET OP!: dataset {dict_foreign_keys[item][0]} wordt niet gecheckt.")


LET OP!: dataset Centraal_monumenten_archief wordt niet gecheckt.
LET OP!: dataset Centraal_monumenten_archief wordt niet gecheckt.
LET OP!: dataset None wordt niet gecheckt.


In [84]:
# test hier of het inladen gelukt is:
# duckdb.sql(f"""DESCRIBE brons_archeologische_terreinen.def_terreinen_v5;""")
# duckdb.sql(f"""DESCRIBE brons_rijksmonumentenregister.tblTEXT_OBJECT;""")

In [85]:
fk_all_match = True
for item in dict_foreign_keys:
    if dict_foreign_keys[item][4]:
        val = duckdb.sql(f"""
        SELECT
            LIST_HAS_ALL((
                SELECT
                    LIST({dict_foreign_keys[item][3]})
                FROM {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}),
                (SELECT
                    LIST({item})
                FROM {table})
        );
        """).fetchone()[0]
        print(f"alle waarden in de foreign key kolom {item} komen voor in de kolom {dict_foreign_keys[item][3]} van {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}: {val}")
        if not val:
            fk_all_match = False
print(f"De foreign key(s) kom(t/en) overeen met een primary key(s) uit de dataset waar deze naar verwijst/verwijzen: {fk_all_match}")
test_resultaten_dim1.append(fk_all_match)


alle waarden in de foreign key kolom terreinnum komen voor in de kolom terreinnum van brons_archeologische_terreinen.def_terreinen_v5: True
alle waarden in de foreign key kolom rijksmonum komen voor in de kolom TXO_TEXT_KEY van brons_rijksmonumentenregister.tblTEXT_OBJECT: False
De foreign key(s) kom(t/en) overeen met een primary key(s) uit de dataset waar deze naar verwijst/verwijzen: False


#### Resultaten dimensie 1

In [86]:
aantal_testen_succes_dim1 = 0

for test in test_resultaten_dim1:
    if test:
        aantal_testen_succes_dim1 += 1

if len(test_resultaten_dim1) == aantal_testen_succes_dim1:
    print(f"Er zijn {len(test_resultaten_dim1)} tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan {aantal_testen_succes_dim1} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 in zijn geheel is voldaan")
    dk_dimensies_voldaan +=1
else:
    print(f"Er zijn {len(test_resultaten_dim1)} tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan {aantal_testen_succes_dim1} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 NIET in zijn geheel is voldaan.")



Er zijn 7 tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan 2 test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 NIET in zijn geheel is voldaan.


## Dimensie 2: Structurele consistentie
* Bevat elke kolom maar 1 datatype, of is dit gemixt?
* Is het datatype voor elke kolom logisch voor de informatie die in die kolom staat?
* Bevat elke kolom maar 1 stuk informatie?
    * Zijn er kolommen waar de eenheid in dezelfde kolom als de waarde staat?
    * Zijn er kolommen met dezelfde naam?
    * Zijn er kolommen waar dezelfde informatie in staat (volgens dezelfde standaard, of een andere?), maar waarvan de kolom niet dezelfde naam heeft?
* Is het gebruik van lege waarden consistent?
* Is het gebruik van onbekende waarden consistent?

## Dimensie 3: Waardenvaliditeit (inhoudelijke correctheid)
* Zijn er kolommen die volgens een bepaalde standaard zijn ingevuld?
    * Zo ja, welke?
    * Zo ja, wordt de standaard ook daadwerkelijk goed gevolgd?
* Zijn er kolommen die niet volgens een standaard zijn ingevuld, waarbij dat wel zou kunnen?
* Komen de waarden overeen met wat er in de kolom zou moeten staan?
* Komen de (tekstuele) waarden binnen een kolom qua informatie en opbouw overeen?
* Zijn er kolommen waarin (bijna) geen data staat?


## Dimensie 4: Granulariteit en eenduidige representatie
* Is het niveau van detail van de data _binnen_ kolommen hetzelfde?
* Zijn er geaggregeerde kolommen (samentrekkingen van 2 kolommen die beiden in de dataset staan)
* Zijn er kolommen waarin andere kolommen met elkaar verrekend zijn? (en zo ja, is dat dan ongewenst?)
* Zijn er kolommen waarbij het niveau van detail niet voldoet voor het beoogde doel?
* Staan de waarden en de indicatie van precisie van deze waarde in losse kolommen?

## Dimensie 5: Semantische eenduidigheid en interoperabiliteit
* Is er een data dictionary?
* Is de data dictionary up-to-date?
* Is er vastgelegd of de kolommen volgens een (standaard) waardenlijst moet worden ingevuld?
* Zijn de referentielijsten expliciet gekoppeld aan de informatie?
* Is voor de waardenlijsten die via linked data (o.i.d.) worden binnengehaald een kolom met uri's aanwezig?
* Zijn er kolommen waar zowel standaard waarden als vrije velden is staan?


## Dimensie 6: Ruimtelijke referentie-eenduidigheid

Alleen van toepassing als het een ruimtelijke dataset betreft (maar let op: ook niet GIS-ready datasets bevatten vaak ruimtelijke informatie)
* Zijn locaties expliciet gemaakt, of zijn er alleen omschrijvingen.
* Is de ruimtelijke informatie binnen kolommen op hetzelfde niveau van detail (als het geen geometrie betreft)
* Is het coordinaatreferentiestelsel gedefinieerd?
*

## Dimensie 7: Ruimtelijke schaal en resolutie consistentie
* Is het schaalniveau van de geometrie gedefinieerd (voor vector datasets)
* Is de resolutie gedefinieerd (voor raster bestanden)
* Heeft de geometrie van de dataset over het gehele bestand hetzelfde schaalniveau/resolutie?
* Past het schaalniveau/de resolutie bij het beoogde doel?
* Komt de precisie van de geometrie overeen met het schaalniveau?

## Dimensie 8: Ruimtelijke geometrische correctheid
* Is het geometrietype expliciet?
* Zijn de geometrieën valide (niet zelfkruisend/niet niet aansluitend/geen gaten)
* Hebben alle geometrieën ook geassocieerde data?

## Dimensie 9: Temporele resolutie
* Zijn er kolommen voor geldigheid?
* Zijn er kolommen voor wanneer een record is toegevoegd?
* Worden "oude" rijen bewaard?
* Gebruiken alle datum kolommen dezelfde vorm? (en zo nee, is dit een probleem?)
* Is eind_geldigheid altijd na begin_geldigheid?
* Zijn eind_geldigheid en begin_geldigheid altijd na registratiedatum?
* Zijn er verschillende kolommen voor verschillende temporele detailniveaus?